In [ ]:
import nico 
from nico import Annotations as sann
from nico import Interactions as sint
from nico import Covariations as scov

import numpy as np
import os
import matplotlib.pyplot as plt 
from matplotlib.collections import PatchCollection

In [ ]:
import igraph
import scanpy as sc
import shutil
import pandas as pd

In [ ]:
adata_TMA_merged_2 = sc.read_h5ad("/cluster3/yflu/STS/TMA/adata_TMA_merged.h5ad")

In [ ]:
sample_counts = adata_TMA_merged_2.obs["Sample"].value_counts()
Samples = sample_counts[sample_counts > 0].index

In [ ]:
Samples = Samples.tolist()

In [ ]:
sample = Samples[0]

In [ ]:
adata = adata_TMA_merged_2[adata_TMA_merged_2.obs["Sample"] == sample].copy()

In [ ]:
adata.write(f'/cluster3/yflu/STS/TMA/separated/{sample}/interaction/{sample}_interact.h5ad')

In [ ]:
adata.write(f"/cluster3/yflu/STS/TMA/separated/{sample}/sct_spatial.h5ad")

In [ ]:
ref_datapath='/cluster3/yflu/STS/TMA/celltypist/'
query_datapath=f'/cluster3/yflu/STS/TMA/separated/{sample}/'
output_nico_dir=f'/cluster3/yflu/STS/TMA/separated/{sample}/interaction/'
output_annotation_dir=None #uses default location
inputRadius=0
#output_annotation_dir=output_nico_dir+'annotations/'
annotation_save_fname= f'{sample}_interact.h5ad'

In [ ]:
ct_counts = adata.obs["Celltype_united"].value_counts()

# 找出小于 50 的 celltype
do_not_use_following_CT_in_niche = ct_counts[ct_counts < 50].index.tolist()

print(do_not_use_following_CT_in_niche)

In [ ]:
niche_pred_output=sint.spatial_neighborhood_analysis(
Radius=inputRadius,
output_nico_dir=output_nico_dir,
anndata_object_name=annotation_save_fname,
spatial_cluster_tag='Celltype_united',
removed_CTs_before_finding_CT_CT_interactions=do_not_use_following_CT_in_niche)

In [ ]:
celltype_niche_interaction_cutoff=0.1

In [ ]:
sint.plot_niche_interactions_with_edge_weight(niche_pred_output,
niche_cutoff=celltype_niche_interaction_cutoff,
saveas="pdf",
transparent_mode=False,
showit=True,
figsize=(10,7),
input_colormap='jet',   #Colormap for node colors, from matplotlib colormaps.
with_labels=True,       #Display cell type labels on the nodes, if True.
node_size=300,          #Size of the nodes.
linewidths=0.5,         #Width of the node border lines.
node_font_size=6,       #Font size for node labels.
alpha=0.5,              #Opacity level for nodes and edges. 1 is fully opaque, and 0 is fully transparent.
font_weight='bold'      #Font weight for node labels; 'bold' for emphasis, 'normal' otherwise.
)

In [ ]:
inputRadius=0
cov_out=scov.gene_covariation_analysis(iNMFmode=True,
    Radius=inputRadius,
    no_of_factors=1,
    refpath=ref_datapath,
    quepath=query_datapath,
    ref_cluster_tag='Celltype_united',
    spatial_integration_modality='double',
    output_niche_prediction_dir=output_nico_dir,
    LRdbFilename='/cluster3/yflu/STS/TMA/NiCoLRdb.txt')

scov.save_LR_interactions_in_excelsheet_and_regression_summary_in_textfile_for_interacting_cell_types(cov_out,
pvalueCutoff=0.05,correlation_with_spearman=True,
LR_plot_NMF_Fa_thres=0.1,LR_plot_Exp_thres=0.1,number_of_top_genes_to_print=5)

scov.find_LR_interactions_in_interacting_cell_types(cov_out,
choose_interacting_celltype_pair=[],
choose_factors_id=[1,1],
pvalueCutoff=0.05,
LR_plot_NMF_Fa_thres=0.1,
LR_plot_Exp_thres=0.1,
saveas="pdf",figsize=(12, 10))

In [ ]:
help(scov.save_LR_interactions_in_excelsheet_and_regression_summary_in_textfile_for_interacting_cell_types)

In [ ]:
for sample in Samples:
    ref_datapath='/cluster3/yflu/STS/TMA/celltypist/'
    query_datapath=f"/cluster3/yflu/STS/TMA/separated/{sample}/"
    output_nico_dir=f'/cluster3/yflu/STS/TMA/separated/{sample}/interaction/'
    output_annotation_dir=None #uses default location
    #output_annotation_dir=output_nico_dir+'annotations/'
    annotation_save_fname= f'{sample}_interact.h5ad'
    inputRadius=0
    cov_out=scov.gene_covariation_analysis(iNMFmode=True,
        Radius=inputRadius,
        no_of_factors=1,
        refpath=ref_datapath,
        ref_cluster_tag='Celltype_united',
        spatial_integration_modality='single',
        output_niche_prediction_dir=output_nico_dir,
        LRdbFilename='/cluster3/yflu/STS/TMA/NiCoLRdb.txt')
    scov.save_LR_interactions_in_excelsheet_and_regression_summary_in_textfile_for_interacting_cell_types(cov_out,
    pvalueCutoff=0.05,correlation_with_spearman=True,
    LR_plot_NMF_Fa_thres=0.1,LR_plot_Exp_thres=0.1,number_of_top_genes_to_print=5)
    scov.find_LR_interactions_in_interacting_cell_types(cov_out,
    choose_interacting_celltype_pair=[],
    choose_factors_id=[1,1],
    pvalueCutoff=0.05,
    LR_plot_NMF_Fa_thres=0.1,
    LR_plot_Exp_thres=0.1,
    saveas="pdf",figsize=(12, 10))

In [ ]:
help(scov.gene_covariation_analysis)